# ADNET — Full Architecture Training, Extended Run (Real Data, Real Model)

**This is the extended-training version**, following up on a completed 20-epoch run that showed a large gap vs. the manuscript's claimed performance (75.44% vs. 99.41% accuracy). This version trains for much longer to test whether more training closes that gap — using the same real architecture (CSPA, HAF, composite loss, oversampling) as before, corrected to use the manuscript's stated moderate oversampling factors (previously fixed after an earlier, more severe bug).

**Built-in time safety net:** Kaggle kills any single session at 12 hours, and we don't know exactly how fast your earlier run went. Rather than guess an epoch count, this notebook tracks wall-clock time directly and stops gracefully — finishing the current epoch, evaluating, and saving a valid result — once it reaches **10.5 hours**, leaving a safety margin. It will use as much of that time as it can to train as many epochs as possible (up to a ceiling of 100), rather than stopping at a fixed, possibly-too-low or possibly-too-high epoch count. If not all 5 folds finish in time, it reports honestly on however many did.

**Setup steps on Kaggle:** identical to the previous run — New Notebook, Add Input (Alzheimer's dataset), Settings → GPU T4 x2 + Internet On, then **Save Version → Save & Run All (Commit)**.

**About your 18 hours of weekly GPU quota:** this run is capped at 10.5 hours to stay under Kaggle's per-session limit, which leaves roughly 7-7.5 hours of your weekly quota unused. If this run completes all 5 folds well within 10.5 hours, that's a clean, complete result. If it hits the time limit partway through, you'll have a partial-but-valid result, and can start a second session on a different day to continue if useful — ask if you want help setting that up once you see how this one goes.

## 1. Setup

In [ ]:
!pip install timm --quiet
import torch, timm, os, glob, random, json
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, ImageEnhance
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type != 'cuda':
    print('WARNING: no GPU detected. Go to Settings -> Accelerator -> GPU T4 x2, then re-run.')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Locate the dataset and remove exact-duplicate copies

Same proven deduplication step as the earlier notebooks — this dataset mirror contains an exact duplicate of every image (12,800 raw -> 6,400 unique).

In [ ]:
print('Contents of /kaggle/input:')
for entry in os.listdir('/kaggle/input'):
    print(' -', entry)

dataset_path = None
for entry in os.listdir('/kaggle/input'):
    candidate = os.path.join('/kaggle/input', entry)
    for root, dirs, files in os.walk(candidate):
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
            dataset_path = candidate
            break
    if dataset_path:
        break

if dataset_path is None:
    raise RuntimeError("No image dataset found under /kaggle/input. Add the Alzheimer's dataset as an input.")
print('\nUsing dataset path:', dataset_path)

class_dirs = []
for root, dirs, files in os.walk(dataset_path):
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        class_dirs.append((root, imgs))

CLASS_NAMES = sorted(set(os.path.basename(d) for d, _ in class_dirs))
print('Classes found:', CLASS_NAMES)
assert len(CLASS_NAMES) == 4, f'Expected 4 classes, found {len(CLASS_NAMES)}: {CLASS_NAMES}'

raw_paths, raw_labels = [], []
for d, imgs in class_dirs:
    cls = os.path.basename(d)
    cls_idx = CLASS_NAMES.index(cls)
    for f in imgs:
        raw_paths.append(os.path.join(d, f))
        raw_labels.append(cls_idx)
print(f'Total raw images found: {len(raw_paths)}')

print('\nDeduplicating exact-duplicate images before splitting...')
!pip install imagehash --quiet
import imagehash

seen_hashes = {}
all_paths_list, all_labels_list = [], []
for path, label in zip(raw_paths, raw_labels):
    try:
        with Image.open(path) as img:
            h = str(imagehash.phash(img, hash_size=16))
    except Exception:
        continue
    if h not in seen_hashes:
        seen_hashes[h] = path
        all_paths_list.append(path)
        all_labels_list.append(label)

all_paths = np.array(all_paths_list)
all_labels = np.array(all_labels_list)
print(f'Unique images after deduplication: {len(all_paths)} (removed {len(raw_paths) - len(all_paths)} duplicates)')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {c}: {(all_labels == i).sum()}')

## 3. MRI-Specific Augmentation (MSA) — reasonable interpretation

The manuscript describes "seven physiologically-grounded augmentation strategies" without giving exact implementation details. This is a reasonable, standard interpretation for MRI-like grayscale medical images: rotation, horizontal flip, contrast/brightness jitter, gamma correction, Gaussian noise, random crop-and-resize. This is disclosed as an interpretation, not a guaranteed match to the original MSA pipeline.

In [ ]:
from torchvision import transforms
import torchvision.transforms.functional as TF

IMG_SIZE = 224

class GammaCorrection:
    def __init__(self, gamma_range=(0.8, 1.2)):
        self.gamma_range = gamma_range
    def __call__(self, img):
        gamma = random.uniform(*self.gamma_range)
        return TF.adjust_gamma(img, gamma)

class GaussianNoise:
    def __init__(self, std=0.02):
        self.std = std
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    GammaCorrection((0.85, 1.15)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.ToTensor(),
    GaussianNoise(0.015),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class MRIDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('L')
        img = self.transform(img)
        return img, self.labels[idx], self.paths[idx]

print('Augmentation pipeline ready.')

## 4. Full ADNET model (SE, CSPA, HAF, composite loss)

This is the code from the Zenodo deposit package's `src/adnet_model.py`, embedded here so this notebook is self-contained. It includes the fixes applied during manuscript review: the HAF module's scale ordering, and a Swin-Transformer channel-format safety check (NHWC vs NCHW).

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.shape
        z = self.gap(x).view(b, c)
        s = self.relu(self.fc1(z))
        s = self.sigmoid(self.fc2(s)).view(b, c, 1, 1)
        return x * s

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = F.adaptive_avg_pool2d(x, 1)
        mx = F.adaptive_max_pool2d(x, 1)
        mc = self.sigmoid(self.mlp(avg) + self.mlp(mx))
        return x * mc

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CSPA(nn.Module):
    def __init__(self, channels, spatial_size=7, reduction=8):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention()
        self.anatomical_prior = nn.Parameter(torch.ones(1, 1, spatial_size, spatial_size) * 0.5)
    def forward(self, x):
        f_prime = self.channel_attn(x)
        m_s = self.spatial_attn(f_prime)
        m_ap = torch.sigmoid(F.interpolate(self.anatomical_prior, size=x.shape[-2:], mode='bilinear', align_corners=False))
        return f_prime * (m_s * m_ap)

class HAF(nn.Module):
    def __init__(self, ch_a1, ch_b1, ch_a2, ch_b2, ch_a3, ch_b3, out_dim=256):
        super().__init__()
        self.fuse1 = nn.Conv2d(ch_a1 + ch_b1, out_dim, kernel_size=1)
        self.fuse2 = nn.Conv2d(ch_a2 + ch_b2 + out_dim, out_dim, kernel_size=1)
        self.fuse3 = nn.Conv2d(ch_a3 + ch_b3 + out_dim, out_dim, kernel_size=1)
        self.gap = nn.AdaptiveAvgPool2d(1)
    def forward(self, fa1, fb1, fa2, fb2, fa3, fb3):
        h1 = self.fuse1(torch.cat([fa1, fb1], dim=1))
        h1_up = F.interpolate(h1, size=fa2.shape[-2:], mode='bilinear', align_corners=False)
        h2 = self.fuse2(torch.cat([fa2, fb2, h1_up], dim=1))
        h2_up = F.interpolate(h2, size=fa3.shape[-2:], mode='bilinear', align_corners=False)
        h3 = self.fuse3(torch.cat([fa3, fb3, h2_up], dim=1))
        return self.gap(h3).flatten(1)

class ClassificationHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_classes=4, dropout=0.4):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    def forward(self, v):
        h = self.dropout(self.relu(self.fc1(v)))
        return self.fc2(h)

class ADNET(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, cspa_reduction=8):
        super().__init__()
        self.stream_a = timm.create_model('efficientnet_b3', pretrained=pretrained, features_only=True, out_indices=(2, 3, 4))
        a_channels = self.stream_a.feature_info.channels()
        try:
            self.stream_b = timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, features_only=True, out_indices=(1, 2, 3), output_fmt='NCHW')
        except TypeError:
            self.stream_b = timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, features_only=True, out_indices=(1, 2, 3))
        b_channels = self.stream_b.feature_info.channels()
        self.cspa_a = CSPA(a_channels[-1], reduction=cspa_reduction)
        self.cspa_b = CSPA(b_channels[-1], reduction=cspa_reduction)
        self.haf = HAF(ch_a1=a_channels[2], ch_b1=b_channels[2], ch_a2=a_channels[1], ch_b2=b_channels[1], ch_a3=a_channels[0], ch_b3=b_channels[0], out_dim=256)
        self.head = ClassificationHead(in_dim=256, num_classes=num_classes)

    @staticmethod
    def _ensure_nchw(feat):
        if feat.dim() != 4:
            return feat
        b, d1, d2, d3 = feat.shape
        looks_nhwc = (d1 == d2) and (d1 != d3)
        looks_nchw = (d2 == d3) and (d2 != d1)
        if looks_nhwc and not looks_nchw:
            return feat.permute(0, 3, 1, 2).contiguous()
        return feat

    def forward(self, x):
        f_shallow_a, f_mid_a, f_deep_a = self.stream_a(x)
        f_shallow_b, f_mid_b, f_deep_b = self.stream_b(x)
        f_shallow_b, f_mid_b, f_deep_b = self._ensure_nchw(f_shallow_b), self._ensure_nchw(f_mid_b), self._ensure_nchw(f_deep_b)
        f_deep_a = self.cspa_a(f_deep_a)
        f_deep_b = self.cspa_b(f_deep_b)
        v = self.haf(f_deep_a, f_deep_b, f_mid_a, f_mid_b, f_shallow_a, f_shallow_b)
        return self.head(v)

class CompositeLoss(nn.Module):
    def __init__(self, class_weights, alpha=0.25, gamma=2.0, lam=0.3, label_smoothing=0.1):
        super().__init__()
        self.register_buffer('class_weights', torch.as_tensor(class_weights, dtype=torch.float32))
        self.alpha = alpha
        self.gamma = gamma
        self.lam = lam
        self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()
        y_onehot = F.one_hot(targets, num_classes).float()
        y_ls = y_onehot * (1 - self.label_smoothing) + self.label_smoothing / num_classes
        wce = -(self.class_weights.unsqueeze(0) * y_ls * log_probs).sum(dim=1).mean()
        pt = (probs * y_onehot).sum(dim=1)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        fl = -(focal_weight * (y_onehot * log_probs).sum(dim=1)).mean()
        return wce + self.lam * fl

print('Full ADNET architecture defined.')

## 5. Training loop — 5-fold CV, oversampling, composite loss, cosine annealing, early stopping

**Important Kaggle note (same as before):** `/kaggle/working` does not persist between separate interactive sessions unless you click **Save Version**. Use **Save & Run All (Commit)** for this long-running cell so it survives you closing the tab, and periodically re-commit if running across multiple days.

Adjust `EPOCHS` below based on how much of your 7 days you want to spend — 20 is a reasonable starting budget (~4-6 hours total); increase it if you have more GPU-hours available.

In [ ]:
import time

EPOCHS = 100              # high ceiling -- the real stopping mechanism is the time budget below, not this number
BATCH_SIZE = 24
N_FOLDS = 5
PATIENCE = 15             # generous, since we now have a much larger time budget to let training actually converge
CKPT_DIR = '/kaggle/working/checkpoints_full_v3'
PRED_DIR = '/kaggle/working/predictions_full_v3'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

# --- Auto-resume from a previous run's completed folds ---
# If you've attached this SAME notebook's own previous output as an input
# (Add Input -> Notebooks -> search this notebook's name -> add its most
# recent version), this copies over any already-completed fold predictions
# so they're skipped here, and only the remaining fold(s) get trained.
# If no matching input is found, this does nothing and all folds train fresh.
import shutil
resumed_count = 0
if os.path.exists('/kaggle/input'):
    for entry in os.listdir('/kaggle/input'):
        candidate = f'/kaggle/input/{entry}'
        for root, dirs, files in os.walk(candidate):
            if root.endswith('predictions_full_v3'):
                for fname in files:
                    if fname.endswith('.json'):
                        src_path = os.path.join(root, fname)
                        dst_path = os.path.join(PRED_DIR, fname)
                        if not os.path.exists(dst_path):
                            shutil.copy(src_path, dst_path)
                            resumed_count += 1
if resumed_count:
    print(f'Resumed {resumed_count} already-completed fold prediction file(s) from a previous run.')
else:
    print('No previous run output found to resume from -- all folds will train fresh.')

# --- Wall-clock safety net ---
# Kaggle kills any single session at 12 hours. Rather than guess an epoch
# count that might run over (losing everything to a hard kill) or under
# (wasting available time), we track elapsed time directly and stop
# gracefully -- finishing the current epoch, evaluating, and saving a
# valid result -- once we approach the limit, with a safety margin.
NOTEBOOK_START_TIME = time.time()
MAX_RUNTIME_HOURS = 10.5   # stop with 1.5 hours of margin below Kaggle's 12-hour cap
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600


def time_remaining():
    return MAX_RUNTIME_SECONDS - (time.time() - NOTEBOOK_START_TIME)


def time_budget_exceeded():
    return time_remaining() <= 0


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def make_oversampled_loader(dataset, labels):
    # Moderate, bounded oversampling matching the manuscript's Section 3.3
    # (8x for Moderate Demented, 2.5x for Mild, 1.5x for Very Mild, 1x for
    # Non-Demented) -- NOT full inverse-class-frequency equalization, which
    # combined with class-weighted loss caused catastrophic overcorrection
    # (majority class near-0% accuracy) in an earlier run of this notebook.
    oversample_factor = {}
    for i, name in enumerate(CLASS_NAMES):
        if 'oderate' in name:
            oversample_factor[i] = 8.0
        elif name.lower().startswith('mild'):
            oversample_factor[i] = 2.5
        elif 'very' in name.lower():
            oversample_factor[i] = 1.5
        else:
            oversample_factor[i] = 1.0

    sample_weights = np.array([oversample_factor[label] for label in labels])
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    return DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)


def train_one_fold(fold_idx, train_idx, test_idx):
    ckpt_path = os.path.join(CKPT_DIR, f'adnet_full_fold{fold_idx}.pt')
    pred_path = os.path.join(PRED_DIR, f'adnet_full_fold{fold_idx}.json')

    if os.path.exists(pred_path):
        print(f'  [skip] fold {fold_idx} already complete')
        return json.load(open(pred_path))

    train_labels = all_labels[train_idx]
    train_ds = MRIDataset(all_paths[train_idx], train_labels, train_tf)
    test_ds = MRIDataset(all_paths[test_idx], all_labels[test_idx], eval_tf)
    train_loader = make_oversampled_loader(train_ds, train_labels)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = ADNET(num_classes=4, pretrained=True).to(device)

    class_counts = np.bincount(train_labels, minlength=4)
    # Softened class weighting: sqrt of inverse-frequency instead of full
    # inverse-frequency, to avoid compounding with the oversampling above
    # (a ~50:1 raw imbalance becomes a ~7:1 weight differential instead).
    class_weights = np.sqrt(len(train_labels) / (4 * np.maximum(class_counts, 1)))
    criterion = CompositeLoss(class_weights=class_weights, alpha=0.25, gamma=2.0, lam=0.3, label_smoothing=0.1).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    start_epoch = 0
    best_f1 = -1.0
    epochs_no_improve = 0
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        opt.load_state_dict(ckpt['opt'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_f1 = ckpt['best_f1']
        print(f'  Resumed fold {fold_idx} from epoch {start_epoch}')

    for epoch in range(start_epoch, EPOCHS):
        if time_budget_exceeded():
            print(f'  [time budget] stopping fold {fold_idx} at epoch {epoch} -- {MAX_RUNTIME_HOURS}h elapsed, saving current state')
            break

        model.train()
        total_loss = 0.0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for imgs, labels, _ in test_loader:
                out = model(imgs.to(device))
                val_preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
                val_true.extend(labels.numpy().tolist())
        val_f1 = f1_score(val_true, val_preds, average='macro')
        print(f'  fold {fold_idx} epoch {epoch+1}/{EPOCHS} loss={total_loss/len(train_loader):.4f} val_f1={val_f1:.4f}')

        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scheduler': scheduler.state_dict(),
                    'epoch': epoch, 'best_f1': max(best_f1, val_f1)}, ckpt_path)

        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path.replace('.pt', '_best.pt'))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'  Early stopping fold {fold_idx} at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
                break

    best_path = ckpt_path.replace('.pt', '_best.pt')
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))

    model.eval()
    all_preds, all_true, all_paths_out = [], [], []
    with torch.no_grad():
        for imgs, labels, paths in test_loader:
            out = model(imgs.to(device))
            all_preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
            all_true.extend(labels.numpy().tolist())
            all_paths_out.extend(list(paths))

    result = {'fold': fold_idx, 'preds': all_preds, 'true': all_true, 'paths': all_paths_out, 'best_val_f1': best_f1}
    json.dump(result, open(pred_path, 'w'))
    del model
    torch.cuda.empty_cache()
    return result


all_results_full = []
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(all_paths, all_labels)):
    if time_budget_exceeded():
        print(f'\n[time budget] {MAX_RUNTIME_HOURS}h elapsed -- not starting fold {fold_idx}.')
        print(f'Completed {len(all_results_full)} of {N_FOLDS} folds within the time budget; reporting on those.')
        break
    print(f'\n=== Fold {fold_idx} === (time remaining: {time_remaining()/3600:.1f}h)')
    result = train_one_fold(fold_idx, train_idx, test_idx)
    all_results_full.append(result)

print(f'\n{len(all_results_full)} of {N_FOLDS} folds completed for full ADNET.')
if len(all_results_full) < N_FOLDS:
    print('NOTE: fewer than 5 folds completed due to the time budget. Results below are valid for the')
    print('folds that did complete, but report this fold count explicitly when sharing results.')

## 6. Overall and per-class results

In [ ]:
accs, f1s = [], []
per_class_correct = {c: 0 for c in range(4)}
per_class_total = {c: 0 for c in range(4)}

for r in all_results_full:
    accs.append(accuracy_score(r['true'], r['preds']))
    f1s.append(f1_score(r['true'], r['preds'], average='macro'))
    for p, t in zip(r['preds'], r['true']):
        per_class_total[t] += 1
        if p == t:
            per_class_correct[t] += 1

print('=' * 70)
print(f'FULL ADNET — REAL {len(all_results_full)}-FOLD CROSS-VALIDATION RESULTS')
print('=' * 70)
print(f'Overall accuracy: {np.mean(accs)*100:.2f}% (+/- {np.std(accs)*100:.2f}%)')
print(f'Overall macro F1: {np.mean(f1s)*100:.2f}% (+/- {np.std(f1s)*100:.2f}%)')
print()
print('Per-class accuracy (pooled across all 5 folds):')
for c, name in enumerate(CLASS_NAMES):
    correct = per_class_correct[c]
    total = per_class_total[c]
    pct = correct / total * 100 if total else 0
    print(f'  {name}: {correct}/{total} = {pct:.2f}%')

from scipy import stats as scipy_stats
mod_idx = [i for i, c in enumerate(CLASS_NAMES) if 'oderate' in c][0]
k, n = per_class_correct[mod_idx], per_class_total[mod_idx]
alpha = 0.05
lo = 0.0 if k == 0 else scipy_stats.beta.ppf(alpha/2, k, n-k+1)
hi = 1.0 if k == n else scipy_stats.beta.ppf(1-alpha/2, k+1, n-k)
print(f'\nModerate Demented, pooled across all 5 folds: {k}/{n} = {k/n*100:.1f}%')
print(f'95% exact (Clopper-Pearson) CI: [{lo*100:.1f}%, {hi*100:.1f}%]')
print()
print('COPY EVERYTHING ABOVE THIS LINE (from the row of = signs) and send it back.')